# Scraping de Plantillas - Transfermarkt ⚽

Este proyecto automatiza la extracción de datos de jugadores desde Transfermarkt utilizando Python.

## 🚀 Flujo de Trabajo

### 1. Preprocesamiento de Enlaces
El script lee un archivo Excel con enlaces básicos de equipos y genera URLs específicas con el parámetro `plus/1`, que permite visualizar información extendida (altura, pie dominante, club anterior, etc.).

### 2. Extracción Automatizada (Selenium + BeautifulSoup)
El motor de scraping realiza las siguientes acciones:
- **Navegación Headless:** Usa Chrome en modo invisible para optimizar recursos.
- **Espera Inteligente:** Implementa `WebDriverWait` para asegurar que la tabla de jugadores cargue antes de intentar leerla.
- **Parsing de Datos:** Extrae información compleja como:
  - Datos básicos (Dorsal, nombre, posición).
  - Información física (Altura, pie dominante).
  - Detalles contractuales (Fin de contrato, fecha de fichaje).
  - Valores de mercado y costes de traspaso.

## 🛠️ Tecnologías Utilizadas
- **Selenium & WebDriver Manager:** Control del navegador.
- **BeautifulSoup4:** Extracción de datos del HTML.
- **Pandas:** Estructuración de datos y exportación a Excel/DataFrames.

In [14]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
from bs4 import BeautifulSoup
import pandas as pd

# URL del equipo
URL = "https://www.transfermarkt.es/rayo-vallecano/kader/verein/367/saison_id/2025/plus/1"

# Configuración de Chrome
options = webdriver.ChromeOptions()
options.add_argument("--headless")
options.add_argument("--disable-gpu")
options.add_argument("--no-sandbox")

# Inicializar driver
service = Service(ChromeDriverManager().install())
driver = webdriver.Chrome(service=service, options=options)

# Abrir página
driver.get(URL)

# Esperar a que la tabla esté cargada
wait = WebDriverWait(driver, 10)
table_element = wait.until(EC.presence_of_element_located((By.XPATH, '//*[@id="yw1"]/table')))

# Extraer HTML de la tabla
html = table_element.get_attribute('outerHTML')
soup = BeautifulSoup(html, 'html.parser')

# Lista para almacenar jugadores
jugadores_data = []

for idx, row in enumerate(soup.select("tbody > tr"), start=1):
    tds = row.find_all("td", recursive=False)

    if len(tds) < 10:
        print(f"⛔ Fila {idx} ignorada (no es jugador)")
        continue

    print(f"\n🧍 JUGADOR #{idx}")
    print("-" * 30)

    # 1️⃣ Número + posición
    numero = ""
    posicion = ""

    num_div = tds[0].find("div", class_="rn_nummer")
    if num_div:
        numero = num_div.get_text(strip=True)

    posicion = tds[0].get("title", "").strip()

    print(f"1️⃣ Número camiseta: {numero}")
    print(f"   Posición: {posicion}")

    # 2️⃣ Nombre
    nombre = ""
    perfil_url = ""

    nombre_td = tds[1].find("td", class_="hauptlink")
    if nombre_td:
        a = nombre_td.find("a")
        if a:
            nombre = a.get_text(strip=True)
            perfil_url = a["href"]

    print(f"2️⃣ Nombre: {nombre}")
    print(f"   Perfil URL: {perfil_url}")

    # 3️⃣ Nacimiento + edad
    nacimiento = ""
    edad = ""

    fecha_edad = tds[2].get_text(strip=True)
    if "(" in fecha_edad:
        nacimiento = fecha_edad.split("(")[0].strip()
        edad = fecha_edad.split("(")[1].replace(")", "").strip()

    print(f"3️⃣ Fecha nacimiento: {nacimiento}")
    print(f"   Edad: {edad}")

    # 4️⃣ Nacionalidades
    nacionalidades = []
    for img in tds[3].find_all("img"):
        if img.get("title"):
            nacionalidades.append(img["title"])

    print(f"4️⃣ Nacionalidades: {', '.join(nacionalidades)}")

    # 5️⃣ Altura
    altura = tds[4].get_text(strip=True)
    print(f"5️⃣ Altura: {altura}")

    # 6️⃣ Pie
    pie = tds[5].get_text(strip=True)
    print(f"6️⃣ Pie dominante: {pie}")

    # 7️⃣ Fecha fichaje
    fecha_fichaje = tds[6].get_text(strip=True)
    print(f"7️⃣ Fecha fichaje: {fecha_fichaje}")

    # 8️⃣ Club anterior + coste
    club_anterior = ""
    coste = ""

    club_img = tds[7].find("img")
    if club_img:
        club_anterior = club_img.get("title", "")

    club_a = tds[7].find("a")
    if club_a and club_a.get("title"):
        title = club_a["title"]
        if ":" in title:
            coste = title.split(":")[1].replace("Ablöse", "").strip()

    print(f"8️⃣ Club anterior: {club_anterior}")
    print(f"   Coste fichaje: {coste}")

    # 9️⃣ Fin contrato
    fin_contrato = tds[8].get_text(strip=True)
    print(f"9️⃣ Fin contrato: {fin_contrato}")

    # 🔟 Valor mercado
    valor_mercado = ""
    valor_a = tds[9].find("a")
    if valor_a:
        valor_mercado = valor_a.get_text(strip=True)

    print(f"🔟 Valor mercado: {valor_mercado}")

    jugador = {
        "Numero": numero,
        "Nombre": nombre,
        "Posicion": posicion,
        "Nacimiento": nacimiento,
        "Edad": edad,
        "Nacionalidades": ", ".join(nacionalidades),
        "Altura": altura,
        "Pie": pie,
        "Fecha_Fichaje": fecha_fichaje,
        "Club_Anterior": club_anterior,
        "Coste_Fichaje": coste,
        "Fin_Contrato": fin_contrato,
        "Valor_Mercado": valor_mercado,
        "Perfil_URL": perfil_url
    }

    jugadores_data.append(jugador)

print("\n✅ LECTURA COMPLETADA")
print(f"📊 Total jugadores procesados: {len(jugadores_data)}")

df = pd.DataFrame(jugadores_data)

driver.quit()

print("\n📈 DATAFRAME FINAL:")
print(df.head())


🧍 JUGADOR #1
------------------------------
1️⃣ Número camiseta: 13
   Posición: Portero
2️⃣ Nombre: Augusto Batalla
   Perfil URL: /augusto-batalla/profil/spieler/275218
3️⃣ Fecha nacimiento: 30/04/1996
   Edad: 29
4️⃣ Nacionalidades: Argentina, Italia
5️⃣ Altura: 1,85m
6️⃣ Pie dominante: Derecho
7️⃣ Fecha fichaje: 01/07/2025
8️⃣ Club anterior: CA River Plate
   Coste fichaje: 1,60 mill. €
9️⃣ Fin contrato: 30/06/2030
🔟 Valor mercado: 6,00 mill. €
⛔ Fila 2 ignorada (no es jugador)
⛔ Fila 3 ignorada (no es jugador)

🧍 JUGADOR #4
------------------------------
1️⃣ Número camiseta: 1
   Posición: Portero
2️⃣ Nombre: Dani Cárdenas
   Perfil URL: /dani-cardenas/profil/spieler/397676
3️⃣ Fecha nacimiento: 28/03/1997
   Edad: 28
4️⃣ Nacionalidades: España
5️⃣ Altura: 1,86m
6️⃣ Pie dominante: Derecho
7️⃣ Fecha fichaje: 18/08/2023
8️⃣ Club anterior: Levante UD
   Coste fichaje: 1,30 mill. €
9️⃣ Fin contrato: 30/06/2027
🔟 Valor mercado: 1,80 mill. €
⛔ Fila 5 ignorada (no es jugador)
⛔ Fila 6 i

In [15]:
df

,Numero,Nombre,Posicion,Nacimiento,Edad,Nacionalidades,Altura,Pie,Fecha_Fichaje,Club_Anterior,Coste_Fichaje,Fin_Contrato,Valor_Mercado,Perfil_URL
0,13,Augusto Batalla,Portero,30/04/1996,29,"Argentina, Italia","1,85m",Derecho,01/07/2025,CA River Plate,"1,60 mill. €",30/06/2030,"6,00 mill. €",/augusto-batalla/profil/spieler/275218
1,1,Dani Cárdenas,Portero,28/03/1997,28,España,"1,86m",Derecho,18/08/2023,Levante UD,"1,30 mill. €",30/06/2027,"1,80 mill. €",/dani-cardenas/profil/spieler/397676
2,32,Nobel Mendy,Defensa,03/09/2004,21,"Senegal, Guinea-Bissau","1,87m",Izquierdo,20/08/2025,Real Betis Balompié,?,30/06/2026,"6,00 mill. €",/nobel-mendy/profil/spieler/1034853
3,16,Abdul Mumin,Defensa,06/06/1998,27,"Ghana, Nigeria","1,88m",Derecho,01/09/2022,Vitória Guimarães SC,"1,50 mill. €",30/06/2026,"4,00 mill. €",/abdul-mumin/profil/spieler/451801
4,5,Luiz Felipe,Defensa,22/03/1997,28,"Italia, Brasil","1,87m",Derecho,07/07/2025,Olympique de Marsella,Libre,30/06/2026,"3,00 mill. €",/luiz-felipe/profil/spieler/457931
5,33,Jozhua Vertrouwd,Defensa,21/08/2004,21,Países Bajos,"1,87m",Izquierdo,12/08/2025,CD Castellón,?,30/06/2026,"3,00 mill. €",/jozhua-vertrouwd/profil/spieler/765467
6,24,Florian Lejeune,Defensa,20/05/1991,34,Francia,"1,90m",Derecho,28/07/2023,Deportivo Alavés,"2,50 mill. €",30/06/2027,"2,00 mill. €",/florian-lejeune/profil/spieler/127108
7,3,Pep Chavarría,Defensa,10/04/1998,27,España,"1,74m",Izquierdo,31/08/2022,Real Zaragoza,"1,80 mill. €",30/06/2030,"10,00 mill. €",/pep-chavarria/profil/spieler/596122
8,22,Alfonso Espino,Defensa,05/01/1992,34,"Uruguay, España","1,72m",Izquierdo,17/07/2023,Cádiz CF,Libre,30/06/2026,"1,00 mill. €",/alfonso-espino/profil/spieler/311044
9,2,Andrei Rațiu,Defensa,20/06/1998,27,"Rumania, España","1,83m",Derecho,26/08/2023,SD Huesca,500 mil €,30/06/2030,"18,00 mill. €",/andrei-ratiu/profil/spieler/527966


In [26]:
import pandas as pd

# Función para transformar el link
def transformar_link(url, temporada=2025):
    # Obtener slug del club
    try:
        slug = url.split("transfermarkt.es/")[1].split("/")[0]
        # Obtener ID del club
        club_id = url.split("verein/")[1].split("/")[0]
        # Construir nuevo link
        nuevo_link = f"https://www.transfermarkt.es/{slug}/kader/verein/{club_id}/saison_id/{temporada}/plus/1"
        return nuevo_link
    except IndexError:
        # En caso de que la URL no tenga el formato esperado
        return ""

# Leer el Excel
df = pd.read_excel("data_notebook/transfermarket_links_equipos.xlsx")  # Cambiar por el nombre de tu archivo

# Crear nueva columna con links transformados
df["Transfermarkt Kader 2025"] = df["Transfermarkt"].apply(transformar_link)

# Guardar nuevo Excel
df.to_excel("data_notebook/transfermarket_links_kader_2025.xlsx", index=False)

print("¡Archivo generado con éxito!")


¡Archivo generado con éxito!


In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
from bs4 import BeautifulSoup
import pandas as pd
import time
import random
import os

# ---------- CONFIGURACIÓN ----------
EXCEL_INPUT = "data_notebook/transfermarket_links_kader_2025.xlsx"       # Excel con columnas: País, Liga, Nombre del Club, Transfermarkt
CSV_OUTPUT = "data_notebook/jugadores_transfermarket.xlsx" \
""
TEMPORADA = 2025                 # Temporada para los links kader
START_INDEX_FILE = "last_index.txt"  # Para reanudar si se interrumpe
# -----------------------------------

# Leer Excel
df_clubs = pd.read_excel(EXCEL_INPUT)

# Crear/continuar archivo CSV
if os.path.exists(CSV_OUTPUT):
    df_jugadores = pd.read_csv(CSV_OUTPUT)
else:
    df_jugadores = pd.DataFrame()

# Determinar desde qué índice continuar
start_idx = 0
if os.path.exists(START_INDEX_FILE):
    with open(START_INDEX_FILE, "r") as f:
        start_idx = int(f.read().strip())

# Configuración de Selenium
options = webdriver.ChromeOptions()
options.add_argument("--headless")
options.add_argument("--disable-gpu")
options.add_argument("--no-sandbox")

service = Service(ChromeDriverManager().install())
driver = webdriver.Chrome(service=service, options=options)

# Función para transformar link startseite -> kader
def transformar_link(url, temporada=TEMPORADA):
    try:
        slug = url.split("transfermarkt.es/")[1].split("/")[0]
        club_id = url.split("verein/")[1].split("/")[0]
        return f"https://www.transfermarkt.es/{slug}/kader/verein/{club_id}/saison_id/{temporada}/plus/1"
    except:
        return None

# Función para procesar un club
def procesar_club(row):
    pais = row["País"]
    liga = row["Liga"]
    club = row["Nombre del Club"]
    url = row["Transfermarkt Kader 2025"]

    url_kader = transformar_link(url)
    if not url_kader:
        return pd.DataFrame([{
            "Pais": pais,
            "Liga": liga,
            "Club": club,
            "Error": "Link no válido"
        }])

    try:
        driver.get(url_kader)
        # Esperar a que la tabla cargue
        wait = WebDriverWait(driver, 10)
        table_element = wait.until(EC.presence_of_element_located((By.XPATH, '//*[@id="yw1"]/table')))
        html = table_element.get_attribute('outerHTML')
        soup = BeautifulSoup(html, 'html.parser')

        jugadores_data = []
        for row_idx, tr in enumerate(soup.select("tbody > tr"), start=1):
            tds = tr.find_all("td", recursive=False)
            if len(tds) < 10:  # Filas que no son jugadores
                continue

            # Extraer datos
            numero = tds[0].find("div", class_="rn_nummer")
            numero = numero.get_text(strip=True) if numero else ""
            posicion = tds[0].get("title", "").strip()
            
            nombre_td = tds[1].find("td", class_="hauptlink")
            if nombre_td and nombre_td.find("a"):
                nombre = nombre_td.find("a").get_text(strip=True)
                perfil_url = nombre_td.find("a")["href"]
            else:
                nombre = perfil_url = ""

            fecha_edad = tds[2].get_text(strip=True)
            if "(" in fecha_edad:
                nacimiento = fecha_edad.split("(")[0].strip()
                edad = fecha_edad.split("(")[1].replace(")", "").strip()
            else:
                nacimiento = fecha_edad
                edad = ""

            nacionalidades = [img["title"] for img in tds[3].find_all("img") if img.get("title")]
            altura = tds[4].get_text(strip=True)
            pie = tds[5].get_text(strip=True)
            fecha_fichaje = tds[6].get_text(strip=True)
            club_anterior = tds[7].find("img").get("title", "") if tds[7].find("img") else ""
            coste = ""
            club_a = tds[7].find("a")
            if club_a and club_a.get("title"):
                title = club_a["title"]
                if ":" in title:
                    coste = title.split(":")[1].replace("Ablöse", "").strip()
            fin_contrato = tds[8].get_text(strip=True)
            valor_mercado = tds[9].find("a").get_text(strip=True) if tds[9].find("a") else ""

            jugadores_data.append({
                "Pais": pais,
                "Liga": liga,
                "Club": club,
                "Numero": numero,
                "Nombre": nombre,
                "Posicion": posicion,
                "Nacimiento": nacimiento,
                "Edad": edad,
                "Nacionalidades": ", ".join(nacionalidades),
                "Altura": altura,
                "Pie": pie,
                "Fecha_Fichaje": fecha_fichaje,
                "Club_Anterior": club_anterior,
                "Coste_Fichaje": coste,
                "Fin_Contrato": fin_contrato,
                "Valor_mercado": valor_mercado,
                "Perfil_URL": perfil_url
            })
        return pd.DataFrame(jugadores_data)

    except Exception as e:
        # Si falla el club, marcar error
        return pd.DataFrame([{
            "Pais": pais,
            "Liga": liga,
            "Club": club,
            "Error": f"No se pudieron obtener jugadores: {e}"
        }])

# ---------- Bucle principal ----------
for idx, row in df_clubs.iloc[start_idx:].iterrows():
    print(f"\nProcesando {row['Nombre del Club']} ({idx})")
    df_result = procesar_club(row)
    df_jugadores = pd.concat([df_jugadores, df_result], ignore_index=True)
    df_jugadores.to_csv(CSV_OUTPUT, index=False)  # Guardar progresivamente
    with open(START_INDEX_FILE, "w") as f:  # Guardar último índice
        f.write(str(idx + 1))
    
    # Espera aleatoria para no parecer un bot
    time.sleep(random.uniform(3, 7))

driver.quit()
print("\n✅ PROCESO COMPLETADO. CSV generado con todos los jugadores.")



Procesando Real Madrid CF (0)

Procesando FC Barcelona (1)

Procesando Atlético de Madrid (2)

Procesando Villarreal CF (3)

Procesando Athletic Club (4)

Procesando Real Sociedad (5)

Procesando Real Betis (6)

Procesando Girona FC (7)

Procesando Valencia CF (8)

Procesando Sevilla FC (9)

Procesando RC Celta de Vigo (10)

Procesando CA Osasuna (11)

Procesando Getafe CF (12)

Procesando RCD Espanyol (13)

Procesando Deportivo Alavés (14)

Procesando Rayo Vallecano (15)

Procesando RCD Mallorca (16)

Procesando Elche CF (17)

Procesando Levante UD (18)

Procesando Real Oviedo (19)

Procesando Manchester City (20)

Procesando Arsenal FC (21)

Procesando Liverpool FC (22)

Procesando Aston Villa (23)

Procesando Tottenham Hotspur (24)

Procesando Chelsea FC (25)

Procesando Newcastle United (26)

Procesando Manchester United (27)

Procesando West Ham United (28)

Procesando Brighton & Hove Albion (29)

Procesando Crystal Palace (30)

Procesando AFC Bournemouth (31)

Procesando Fulham 

In [3]:
"""
transfermarkt_links_scraper.py
==============================

Lee:
- equipos_unicos.txt
- jugadores_unicos.txt

Busca en Transfermarkt:
- URL del perfil del club y logo
- URL del perfil del jugador e imagen

Guarda:
- equipos_transfermarkt.csv
- jugadores_transfermarkt.csv

Requisitos:
    pip install requests beautifulsoup4 pandas lxml rapidfuzz

Uso:
    python transfermarkt_links_scraper.py
"""

from __future__ import annotations

import csv
import logging
import random
import re
import time
import unicodedata
from pathlib import Path
from urllib.parse import quote, urljoin

import pandas as pd
import requests
from bs4 import BeautifulSoup
from rapidfuzz import fuzz


# ============================================================================
# CONFIG
# ============================================================================

BASE_DIR = Path(".")
EQUIPOS_TXT = BASE_DIR / "equipos_unicos.txt"
JUGADORES_TXT = BASE_DIR / "jugadores_unicos.txt"

EQUIPOS_CSV = BASE_DIR / "equipos_transfermarkt.csv"
JUGADORES_CSV = BASE_DIR / "jugadores_transfermarkt.csv"

REQUEST_DELAY_MIN = 1.2
REQUEST_DELAY_MAX = 2.8
TIMEOUT = 25

TRANSFERMARKT_BASE = "https://www.transfermarkt.com"
SEARCH_URL = "https://www.transfermarkt.com/schnellsuche/ergebnis/schnellsuche"

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/137.0.0.0 Safari/537.36"
    ),
    "Accept-Language": "en-US,en;q=0.9,es;q=0.8",
}


# ============================================================================
# LOGGING
# ============================================================================

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
)
logger = logging.getLogger("transfermarkt-scraper")


# ============================================================================
# HELPERS
# ============================================================================

def sleep_random():
    time.sleep(random.uniform(REQUEST_DELAY_MIN, REQUEST_DELAY_MAX))


def fix_text(value: str) -> str:
    if value is None:
        return ""
    s = str(value).strip()

    replacements = {
        "Ã¡": "á", "Ã©": "é", "Ã­": "í", "Ã³": "ó", "Ãº": "ú",
        "Ã±": "ñ", "Ã¼": "ü", "Ã": "Á", "Ã‰": "É", "Ã": "Í",
        "Ã“": "Ó", "Ãš": "Ú", "Ã‘": "Ñ", "Ãœ": "Ü", "â‚¬": "€", "Â": "",
    }
    for bad, good in replacements.items():
        s = s.replace(bad, good)

    return s.strip()


def normalize_text(value: str) -> str:
    s = fix_text(value).lower()
    s = unicodedata.normalize("NFD", s)
    s = "".join(c for c in s if unicodedata.category(c) != "Mn")
    s = re.sub(r"\s+", " ", s).strip()
    return s


def read_txt_lines(path: Path) -> list[str]:
    if not path.exists():
        raise FileNotFoundError(f"No existe el archivo: {path}")

    lines = path.read_text(encoding="utf-8").splitlines()
    lines = [fix_text(x).strip() for x in lines if x.strip()]
    logger.info("Leídas %d líneas desde %s", len(lines), path.name)
    return lines


def load_existing_csv(path: Path, key_col: str) -> dict[str, dict]:
    if not path.exists():
        return {}

    df = pd.read_csv(path, encoding="utf-8")
    if key_col not in df.columns:
        return {}

    out = {}
    for _, row in df.iterrows():
        out[str(row[key_col])] = row.to_dict()

    logger.info("Cargadas %d filas existentes de %s", len(out), path.name)
    return out


def save_rows_csv(path: Path, rows: list[dict], columns: list[str]):
    df = pd.DataFrame(rows)
    for col in columns:
        if col not in df.columns:
            df[col] = None
    df = df[columns]
    df.to_csv(path, index=False, encoding="utf-8-sig")
    logger.info("CSV guardado: %s (%d filas)", path.name, len(df))


def best_image_from_tag(tag) -> str | None:
    if tag is None:
        return None

    for attr in ["src", "data-src", "data-lazy", "data-lazy-src"]:
        val = tag.get(attr)
        if val and str(val).strip():
            return str(val).strip()

    return None


# ============================================================================
# HTTP
# ============================================================================

session = requests.Session()
session.headers.update(HEADERS)


def fetch(url: str, params: dict | None = None) -> requests.Response | None:
    try:
        logger.info("GET %s params=%s", url, params)
        response = session.get(url, params=params, timeout=TIMEOUT)
        logger.info("Status %s para %s", response.status_code, response.url)

        if response.status_code == 200:
            sleep_random()
            return response

        logger.warning("Respuesta no OK: %s", response.status_code)
        sleep_random()
        return None

    except Exception as e:
        logger.exception("Error en request %s: %s", url, e)
        sleep_random()
        return None


# ============================================================================
# SEARCH TRANSFERMARKT
# ============================================================================

def transfermarkt_search(query: str) -> BeautifulSoup | None:
    response = fetch(SEARCH_URL, params={"query": query})
    if not response:
        return None
    return BeautifulSoup(response.text, "lxml")


def score_candidate(input_name: str, candidate_name: str) -> int:
    return fuzz.token_sort_ratio(normalize_text(input_name), normalize_text(candidate_name))


def make_absolute(url: str | None) -> str | None:
    if not url:
        return None
    return urljoin(TRANSFERMARKT_BASE, url)


# ============================================================================
# TEAM SCRAPING
# ============================================================================

def parse_team_search_results(soup: BeautifulSoup, team_name: str) -> list[dict]:
    candidates = []

    # Buscar links que apunten a /verein/
    for a in soup.select('a[href*="/verein/"]'):
        href = a.get("href")
        text = fix_text(a.get_text(" ", strip=True))

        if not href or not text:
            continue

        full_url = make_absolute(href)
        score = score_candidate(team_name, text)

        # intentar sacar imagen cercana
        logo_url = None
        img = None

        # imagen dentro del mismo link
        img = a.find("img")
        if img:
            logo_url = best_image_from_tag(img)

        # o dentro del parent row/cell
        if not logo_url:
            parent = a.parent
            if parent:
                img = parent.find("img")
                if img:
                    logo_url = best_image_from_tag(img)

        candidates.append({
            "name_found": text,
            "url": full_url,
            "image_url": logo_url,
            "score": score,
        })

    # deduplicar por url
    dedup = {}
    for c in candidates:
        if c["url"] not in dedup or c["score"] > dedup[c["url"]]["score"]:
            dedup[c["url"]] = c

    out = sorted(dedup.values(), key=lambda x: x["score"], reverse=True)
    logger.info("Candidatos club para '%s': %d", team_name, len(out))
    return out


def enrich_team_page(team_url: str) -> dict:
    response = fetch(team_url)
    if not response:
        return {"team_logo_url": None}

    soup = BeautifulSoup(response.text, "lxml")

    logo_url = None

    # Transfermarkt suele tener imágenes del club en img tags relevantes
    for img in soup.find_all("img"):
        src = best_image_from_tag(img)
        alt = fix_text(img.get("alt", ""))

        if src and alt:
            if "logo" in alt.lower() or "wappen" in alt.lower():
                logo_url = src
                break

    # fallback: primer escudo razonable
    if not logo_url:
        for img in soup.find_all("img"):
            src = best_image_from_tag(img)
            if src and "/tiny/" in src or src and "/header/" in src:
                logo_url = src
                break

    return {
        "team_logo_url": logo_url
    }


def scrape_team(team_name: str) -> dict:
    logger.info("=" * 80)
    logger.info("Procesando equipo: %s", team_name)

    soup = transfermarkt_search(team_name)
    if not soup:
        return {
            "team_name": team_name,
            "transfermarkt_team_url": None,
            "team_logo_url": None,
            "status": "search_failed",
            "notes": "No response from search",
        }

    candidates = parse_team_search_results(soup, team_name)

    if not candidates:
        return {
            "team_name": team_name,
            "transfermarkt_team_url": None,
            "team_logo_url": None,
            "status": "not_found",
            "notes": "No team candidates found",
        }

    best = candidates[0]
    details = enrich_team_page(best["url"])

    status = "matched"
    notes = f"best_score={best['score']}"
    if len(candidates) > 1:
        notes += f"; candidates={len(candidates)}"

    row = {
        "team_name": team_name,
        "name_found": best["name_found"],
        "transfermarkt_team_url": best["url"],
        "team_logo_url": details.get("team_logo_url") or best.get("image_url"),
        "match_score": best["score"],
        "status": status,
        "notes": notes,
    }

    logger.info("Equipo resuelto: %s -> %s", team_name, row["transfermarkt_team_url"])
    return row


# ============================================================================
# PLAYER SCRAPING
# ============================================================================

def parse_player_search_results(soup: BeautifulSoup, player_name: str) -> list[dict]:
    candidates = []

    # Buscar links que apunten a /profil/spieler/ o /spieler/
    for a in soup.select('a[href*="/profil/spieler/"], a[href*="/spieler/"]'):
        href = a.get("href")
        text = fix_text(a.get_text(" ", strip=True))

        if not href or not text:
            continue

        # filtrar falsos positivos muy cortos
        if len(text) < 3:
            continue

        full_url = make_absolute(href)
        score = score_candidate(player_name, text)

        player_image_url = None
        img = a.find("img")
        if img:
            player_image_url = best_image_from_tag(img)

        if not player_image_url:
            parent = a.parent
            if parent:
                img = parent.find("img")
                if img:
                    player_image_url = best_image_from_tag(img)

        candidates.append({
            "name_found": text,
            "url": full_url,
            "image_url": player_image_url,
            "score": score,
        })

    # deduplicar
    dedup = {}
    for c in candidates:
        if c["url"] not in dedup or c["score"] > dedup[c["url"]]["score"]:
            dedup[c["url"]] = c

    out = sorted(dedup.values(), key=lambda x: x["score"], reverse=True)
    logger.info("Candidatos jugador para '%s': %d", player_name, len(out))
    return out


def enrich_player_page(player_url: str) -> dict:
    response = fetch(player_url)
    if not response:
        return {"player_image_url": None}

    soup = BeautifulSoup(response.text, "lxml")

    player_image_url = None

    # buscar imagen principal razonable
    for img in soup.find_all("img"):
        src = best_image_from_tag(img)
        alt = fix_text(img.get("alt", ""))

        if src and alt:
            # a menudo la foto principal contiene el nombre del jugador
            if len(alt) > 2:
                player_image_url = src
                break

    return {
        "player_image_url": player_image_url
    }


def scrape_player(player_name: str) -> dict:
    logger.info("=" * 80)
    logger.info("Procesando jugador: %s", player_name)

    soup = transfermarkt_search(player_name)
    if not soup:
        return {
            "player_name": player_name,
            "transfermarkt_player_url": None,
            "player_image_url": None,
            "status": "search_failed",
            "notes": "No response from search",
        }

    candidates = parse_player_search_results(soup, player_name)

    if not candidates:
        return {
            "player_name": player_name,
            "transfermarkt_player_url": None,
            "player_image_url": None,
            "status": "not_found",
            "notes": "No player candidates found",
        }

    best = candidates[0]
    details = enrich_player_page(best["url"])

    status = "matched"
    notes = f"best_score={best['score']}"
    if len(candidates) > 1:
        notes += f"; candidates={len(candidates)}"

    row = {
        "player_name": player_name,
        "name_found": best["name_found"],
        "transfermarkt_player_url": best["url"],
        "player_image_url": details.get("player_image_url") or best.get("image_url"),
        "match_score": best["score"],
        "status": status,
        "notes": notes,
    }

    logger.info("Jugador resuelto: %s -> %s", player_name, row["transfermarkt_player_url"])
    return row


# ============================================================================
# PIPELINES
# ============================================================================

def process_teams():
    teams = read_txt_lines(EQUIPOS_TXT)
    existing = load_existing_csv(EQUIPOS_CSV, "team_name")

    rows = list(existing.values())
    done = set(existing.keys())

    columns = [
        "team_name",
        "name_found",
        "transfermarkt_team_url",
        "team_logo_url",
        "match_score",
        "status",
        "notes",
    ]

    total = len(teams)
    for idx, team_name in enumerate(teams, start=1):
        if team_name in done:
            logger.info("[EQUIPOS] Saltando ya procesado (%d/%d): %s", idx, total, team_name)
            continue

        logger.info("[EQUIPOS] %d/%d -> %s", idx, total, team_name)
        row = scrape_team(team_name)
        rows.append(row)

        save_rows_csv(EQUIPOS_CSV, rows, columns)

    logger.info("Proceso de equipos completado.")


def process_players():
    players = read_txt_lines(JUGADORES_TXT)
    existing = load_existing_csv(JUGADORES_CSV, "player_name")

    rows = list(existing.values())
    done = set(existing.keys())

    columns = [
        "player_name",
        "name_found",
        "transfermarkt_player_url",
        "player_image_url",
        "match_score",
        "status",
        "notes",
    ]

    total = len(players)
    for idx, player_name in enumerate(players, start=1):
        if player_name in done:
            logger.info("[JUGADORES] Saltando ya procesado (%d/%d): %s", idx, total, player_name)
            continue

        logger.info("[JUGADORES] %d/%d -> %s", idx, total, player_name)
        row = scrape_player(player_name)
        rows.append(row)

        save_rows_csv(JUGADORES_CSV, rows, columns)

    logger.info("Proceso de jugadores completado.")


# ============================================================================
# MAIN
# ============================================================================

def main():
    logger.info("Inicio del scraping Transfermarkt")
    process_teams()
    process_players()
    logger.info("Scraping finalizado")


if __name__ == "__main__":
    main()

2026-06-12 18:45:36,007 | INFO | Inicio del scraping Transfermarkt
2026-06-12 18:45:36,076 | INFO | Leídas 185 líneas desde equipos_unicos.txt
2026-06-12 18:45:36,088 | INFO | [EQUIPOS] 1/185 -> 1. FC Heidenheim
2026-06-12 18:45:36,338 | INFO | ================================================================================
2026-06-12 18:45:36,361 | INFO | Procesando equipo: 1. FC Heidenheim
2026-06-12 18:45:36,367 | INFO | GET https://www.transfermarkt.com/schnellsuche/ergebnis/schnellsuche params={'query': '1. FC Heidenheim'}
2026-06-12 18:45:51,579 | INFO | Status 200 para https://www.transfermarkt.com/schnellsuche/keinergebnis/schnellsuche?query=1.+FC+Heidenheim


FeatureNotFound: Couldn't find a tree builder with the features you requested: lxml. Do you need to install a parser library?